In [ ]:
import numpy as np
import pandas as pd


def parse_annotations(raw):
    """
    解析 raw.annotations，返回结构化 DataFrame。

    Parameters
    ----------
    raw : mne.io.Raw
        包含 annotations 的 Raw 对象。

    Returns
    -------
    df : pd.DataFrame
        包含列: onset, duration, description, trial_index
    event_id : dict
        描述到事件ID的映射
    events : np.ndarray
        MNE 格式的 events 数组 (n_events, 3)
    """
    # ---- 1. 提取 annotations 到 DataFrame ----
    annot = raw.annotations
    df = pd.DataFrame({
        'onset':      annot.onset,
        'duration':   annot.duration,
        'description': annot.description,
    })

    # 重置 trial 编号（从 0 开始）
    df['trial_index'] = df.index

    # ---- 2. 生成 event_id 映射 ----
    unique_labels = sorted(df['description'].unique())
    event_id = {label: i + 1 for i, label in enumerate(unique_labels)}
    # 例: {'feet': 1, 'left_hand': 2, 'right_hand': 3, 'tongue': 4}

    # ---- 3. 构造 MNE events 数组 ----
    # events: (n_events, 3) -> [sample, 0, event_id]
    events, _ = mne.events_from_annotations(raw, event_id=event_id)

    # ---- 4. 统计信息 ----
    print("=" * 50)
    print(f"总 trial 数: {len(df)}")
    print(f"事件类别:   {list(event_id.keys())}")
    print("-" * 50)
    print(df['description'].value_counts().to_string())
    print("=" * 50)

    return df, event_id, events


In [29]:
import os
from collections import Counter
import pandas as pd

import mne
import numpy as np
from moabb.datasets import BNCI2014_001

# 数据集
data_dir = os.path.abspath("./../datasets")
mne.set_config("MNE_DATA", data_dir)
dataset = BNCI2014_001()
dataset.download()

subjects = dataset.get_data(subjects=[1])
raw = subjects[1]['0train']['0'] 

# 解析标注
annot = raw.annotations
df = pd.DataFrame({
    'onset':      annot.onset,
    'duration':   annot.duration,
    'description': annot.description,
})
df['trial_index'] = df.index
print(df)

# event_label
unique_labels = sorted(df['description'].unique())
print(f"event_label: {unique_labels}")

# event_id
for i, label in enumerate(unique_labels):
    print(f"{label:12s}: {i + 1}")
event_id = {label: i + 1 for i, label in enumerate(unique_labels)}
print(f"event_id: {event_id}")

# events
events, event_dict = mne.events_from_annotations(raw, event_id=event_id)
df_event = pd.DataFrame(events)
print(f"事件数: {len(df_event)}")
print(f"事件信息形状: {df_event.shape}") # (事件数, 3): (sample：第几个采样点, previous：, event_id)
df_event





# 查看前几行
# df.head(10)


      onset  duration description  trial_index
0     3.000       4.0      tongue            0
1    11.012       4.0        feet            1
2    18.684       4.0  right_hand            2
3    26.492       4.0   left_hand            3
4    34.524       4.0   left_hand            4
5    42.968       4.0  right_hand            5
6    50.636       4.0        feet            6
7    58.836       4.0      tongue            7
8    66.560       4.0  right_hand            8
9    74.552       4.0        feet            9
10   82.176       4.0   left_hand           10
11   89.756       4.0   left_hand           11
12   97.644       4.0   left_hand           12
13  105.420       4.0      tongue           13
14  113.288       4.0  right_hand           14
15  121.768       4.0  right_hand           15
16  129.800       4.0   left_hand           16
17  138.064       4.0   left_hand           17
18  146.208       4.0        feet           18
19  154.472       4.0   left_hand           19
20  162.752  

,0,1,2
0,750,0,4
1,2753,0,1
2,4671,0,3
3,6623,0,2
4,8631,0,2
5,10742,0,3
6,12659,0,1
7,14709,0,4
8,16640,0,3
9,18638,0,1


In [26]:

# 用 event_id 后续做 epoch
epochs = mne.Epochs(raw, events, event_id=event_id,
                    tmin=-0.2, tmax=3.0, baseline=(None, 0),
                    picks='eeg', preload=True)

print(epochs.info)


Not setting metadata
48 matching events found
Setting baseline interval to [-0.2, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 48 events and 801 original time points ...
0 bad epochs dropped
<Info | 11 non-empty values
 bads: []
 ch_names: Fz, FC3, FC1, FCz, FC2, FC4, C5, C3, C1, Cz, C2, C4, C6, CP3, ...
 chs: 22 EEG
 custom_ref_applied: False
 description: Artifacts: 2/48 trials;
 dig: 25 items (3 Cardinal, 22 EEG)
 highpass: 0.0 Hz
 line_freq: 50.0
 lowpass: 125.0 Hz
 meas_date: 2008-01-01 00:00:00 UTC
 nchan: 22
 projs: []
 sfreq: 250.0 Hz
 subject_info: <subject_info | birthday: 1986-01-01, sex: 2, hand: 1, his_id: sub-01>
>


In [30]:
parse_annotations(raw)

    onset  duration description  trial_index
0   3.000       4.0      tongue            0
1  11.012       4.0        feet            1
2  18.684       4.0  right_hand            2
3  26.492       4.0   left_hand            3
4  34.524       4.0   left_hand            4
5  42.968       4.0  right_hand            5
6  50.636       4.0        feet            6
7  58.836       4.0      tongue            7
8  66.560       4.0  right_hand            8
9  74.552       4.0        feet            9
Used Annotations descriptions: [np.str_('feet'), np.str_('left_hand'), np.str_('right_hand'), np.str_('tongue')]
总 trial 数: 48
事件类别:   ['feet', 'left_hand', 'right_hand', 'tongue']
--------------------------------------------------
description
tongue        12
feet          12
right_hand    12
left_hand     12


(      onset  duration description  trial_index
 0     3.000       4.0      tongue            0
 1    11.012       4.0        feet            1
 2    18.684       4.0  right_hand            2
 3    26.492       4.0   left_hand            3
 4    34.524       4.0   left_hand            4
 5    42.968       4.0  right_hand            5
 6    50.636       4.0        feet            6
 7    58.836       4.0      tongue            7
 8    66.560       4.0  right_hand            8
 9    74.552       4.0        feet            9
 10   82.176       4.0   left_hand           10
 11   89.756       4.0   left_hand           11
 12   97.644       4.0   left_hand           12
 13  105.420       4.0      tongue           13
 14  113.288       4.0  right_hand           14
 15  121.768       4.0  right_hand           15
 16  129.800       4.0   left_hand           16
 17  138.064       4.0   left_hand           17
 18  146.208       4.0        feet           18
 19  154.472       4.0   left_hand      